In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import pathlib
from functools import partial

import time
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['text.usetex'] = True
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}"

In [ ]:
import jax
import jax.numpy as jnp
import equinox as eqx
import diffrax

import gpjax as gpx

In [ ]:
cols = mpl.rcParams["axes.prop_cycle"].by_key()["color"]

In [ ]:
from admpe.utils.helper import get_plot_labels
from admpe.data_management import (
    list_excitation_results,
    provide_excitation_results_overview,
)

from admpe.evaluation.excitation_evaluation import (
    load_excitation_comparison,
    load_corresponding_modeling_results,
)
from admpe.evaluation.excitation_visualization import (
    plot_excitation_feature_grid,
    compare_excitation_feature_grids,
    shared_density_levels,
)
from admpe.evaluation.modeling_evaluation import load_testing_sequences
from admpe.evaluation.shared import visualize_rollout_on_density

#### Configuration:

In [ ]:
SYSTEM = "cart_pole"
EXPERIMENT = "noise_1_e_minus_6"
MODEL_TYPE = "neural_euler"
EXCITATION_SEED = 6
N_CONTOUR_LEVELS = 11
CONTOUR_LEVELS_PER_PAIR = True  

OBJECTIVES = (
    "dmpe_uniform",
    "dmpe_static_dynamics",
    "dmpe_static_mixture_dynamics",
)

LABELS = get_plot_labels(SYSTEM)

#### Gather experiments:

In [ ]:
entries = list_excitation_results(
    system_name=SYSTEM,
    exp_name=EXPERIMENT,
    model_type=MODEL_TYPE,
    excitation_objective=list(OBJECTIVES),
    status="completed",
)
if not entries:
    raise ValueError("No completed runs match the configuration.")

excitation_results_overview = provide_excitation_results_overview(entries)

display(
    pd.crosstab(
        excitation_results_overview["objective"], excitation_results_overview["seed"]
    ).reindex(OBJECTIVES, fill_value=0)
)

#### Select runs:

In [ ]:
runs = load_excitation_comparison(
    excitation_results_overview,
    objectives=OBJECTIVES,
    seed=EXCITATION_SEED,
)

CONTOUR_LEVELS = shared_density_levels(
    runs,
    view="achieved",
    n_levels=N_CONTOUR_LEVELS,
    per_pair=CONTOUR_LEVELS_PER_PAIR,
)

TARGET_CONTOUR_LEVELS = shared_density_levels(
    runs,
    view="target",
    n_levels=N_CONTOUR_LEVELS,
    per_pair=CONTOUR_LEVELS_PER_PAIR,
)

display(
    excitation_results_overview.loc[
        excitation_results_overview["seed"].eq(EXCITATION_SEED),
        [
            "objective", "seed", "name", "n_time_steps",
            "bandwidth", "points_per_dim", "elapsed_seconds",
        ],
    ]
)

#### Comparison data:

In [ ]:
VIEW = "target"  # "samples", "target", or "achieved"

for objective, result in runs.items():
    fig = plot_excitation_feature_grid(
        result,
        view=VIEW,
        labels=LABELS["both"],
        levels=TARGET_CONTOUR_LEVELS if VIEW == "target" else CONTOUR_LEVELS,
    )
    fig.suptitle(
        f"{objective.replace('_', ' ')} | "
        f"{VIEW} | excitation seed {EXCITATION_SEED}"
    )
    fig.tight_layout()
    display(fig)
    plt.close(fig)

In [ ]:
VIEW = "target"

for fig, axes in compare_excitation_feature_grids(
    runs,
    view=VIEW,
    title=f"{VIEW} | excitation seed {EXCITATION_SEED}",
    labels=LABELS["both"],
    levels=TARGET_CONTOUR_LEVELS if VIEW == "target" else CONTOUR_LEVELS,
):
    display(fig)
    plt.close(fig)

#### Comparison rollouts:

In [ ]:
MODEL_SEED = 1
matched_modeling_results = {}

for objective, run in runs.items():
    candidates = load_corresponding_modeling_results(
        run,
        model_type=MODEL_TYPE,
        exp_name=EXPERIMENT,
    )
    matches = [
        result for result in candidates
        if result["metadata"]["seed"] == MODEL_SEED
    ]
    if len(matches) != 1:
        raise ValueError(
            f"{objective}: expected one model with seed "
            f"{MODEL_SEED}, found {len(matches)}."
        )
    matched_modeling_results[objective] = matches[0]

In [ ]:
for objective, excitation_result in runs.items():

    modeling_result = matched_modeling_results[objective]

    testing = load_testing_sequences(
        excitation_result["metadata"]["system_name"]
    )
    test_data = testing["aprbs"]
    
    for index in [0]:
        fig, axes = visualize_rollout_on_density(
            excitation_result,
            modeling_result,
            test_data["observations"][index],
            test_data["actions"][index],
            max_steps=200,
            labels=LABELS["both"],
            levels=CONTOUR_LEVELS,
        )
        fig.suptitle(objective, fontsize=20)
        plt.show()

In [ ]:
from admpe.evaluation.shared import compare_rollouts_on_density

SIGNAL = "multisine"
test_data = load_testing_sequences(SYSTEM)[SIGNAL]

for index in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    for fig, axes in compare_rollouts_on_density(
        runs,
        matched_modeling_results,
        test_data["observations"][index],
        test_data["actions"][index],
        labels=LABELS["both"],
        levels=CONTOUR_LEVELS,
        max_steps=200,
        rows_per_figure=12,
        title=(
            f"{SIGNAL.upper()} trajectory {index} | "
            f"excitation seed {EXCITATION_SEED} | "
            f"model seed {MODEL_SEED}"
        ),
    ):
        display(fig)
        plt.close(fig)

In [ ]:
from admpe.evaluation.shared import compare_rollouts_on_density

SIGNAL = "aprbs"
test_data = load_testing_sequences(SYSTEM)[SIGNAL]

for index in [0, 1, 2, 3, 4, 5, 6, 7, 8]:
    for fig, axes in compare_rollouts_on_density(
        runs,
        matched_modeling_results,
        test_data["observations"][index],
        test_data["actions"][index],
        labels=LABELS["both"],
        levels=CONTOUR_LEVELS,
        max_steps=200,
        rows_per_figure=12,
        title=(
            f"{SIGNAL.upper()} trajectory {index} | "
            f"excitation seed {EXCITATION_SEED} | "
            f"model seed {MODEL_SEED}"
        ),
    ):
        display(fig)
        plt.close(fig)

In [ ]:
SIGNAL = "ocp"
test_data = load_testing_sequences(SYSTEM)[SIGNAL]

for index in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    for fig, axes in compare_rollouts_on_density(
        runs,
        matched_modeling_results,
        test_data["observations"][index],
        test_data["actions"][index],
        labels=LABELS["both"],
        levels=CONTOUR_LEVELS,
        max_steps=200,
        rows_per_figure=12,
        title=(
            f"{SIGNAL.upper()} trajectory {index} | "
            f"excitation seed {EXCITATION_SEED} | "
            f"model seed {MODEL_SEED}"
        ),
    ):
        display(fig)
        plt.close(fig)